Kasus 3: Klasifikasi Spesies Penguin Antarktika

1. Audit Data & Temuan Feature Engineering
Dataset kasus03_penguins.csv terdiri dari 344 baris pengamatan penguin dari tiga pulau di Antarktika dengan 8 kolom dan 0 baris duplikat. Target klasifikasi (species) terbagi ke dalam tiga kelas: Adelie (152 data), Gentoo (124 data), dan Chinstrap (68 data).

Keputusan Pembersihan & Rekayasa Fitur (fitur.py):Penanganan Missing Value:
- Ditemukan nilai kosong pada sex (11 baris) serta masing-masing 2 baris pada bill_length_mm, bill_depth_mm, flipper_length_mm, dan body_mass_g. Untuk mencegah data leakage, dataset dibagi menjadi 80% data latih (275 baris) dan 20% data uji (69 baris) secara stratified sebelum proses imputasi. Statistik imputasi dihitung murni dari data latih menggunakan median untuk fitur numerik (tahan terhadap pencilan/outlier) dan modus (most frequent) untuk fitur kategorik (sex dan island).
- Pembuatan Fitur Baru (bill_ratio): Dibuat fitur rasio paruh melalui formula $\text{bill\_ratio} = \frac{\text{bill\_length\_mm}}{\text{bill\_depth\_mm}}$. Secara morfologis, rasio proporsi panjang terhadap ketebalan paruh memberikan pemisahan yang jauh lebih tajam antarspesies dibandingkan ukuran absolut secara terpisah.   
- Penghapusan Fitur (year): Kolom year (tahun pengamatan 2007–2009) dibuang dari pemodelan. Tahun pencatatan tidak memiliki hubungan kausal biologis terhadap identitas spesies penguin; mempertahankannya justru berisiko membuat model menghafal pola waktu pengamatan (overfitting).

2. Perbandingan Model KlasifikasiPemilihan model dilakukan menggunakan metode 5-Fold Cross-Validation pada data latih (X_train) untuk membandingkan performa dua algoritma yang terbungkus dalam pipeline preprocessing (StandardScaler untuk numerik dan OneHotEncoder untuk kategorik):

- Algoritma KlasifikasiMean CV Accuracy (5-Fold)Standar Deviasi CVMean CV F1-MacroStatusLogistic Regression 99,27% (0.9927)$\pm 0.0089$0.9924 Model Terpilih

- Random Forest Classifier 98,91% (0.9891)$\pm 0.0089 $0.9894 Pembanding

Berdasarkan hasil validasi silang, Logistic Regression unggul tipis dengan rata-rata akurasi 99,27% dan stabilitas tinggi, sehingga dipilih sebagai model final untuk dilatih ulang pada seluruh data latih.

3. Hasil Metrik Evaluasi (Data Test)
Model final dievaluasi satu kali pada 69 sampel data uji (X_test) yang belum pernah dilihat selama proses pelatihan:

- Accuracy: 1.000 (100,0%)

- Balanced Accuracy: 1.000 (100,0%)

- Macro F1-Score: 1.000 (100,0%)

- Weighted F1-Score: 1.000 (100,0%)

Rincian Classification Report & Confusion Matrix:
Kelas Spesies Precision RecallF1-Score Jumlah Data Uji (Support) Prediksi Benar
- Adelie 1.000 1.000 1.000 30 30 (0 salah)
- Chinstrap 1.000 1.000 1.000 14 14 (0 salah)
- Gentoo 1.000 1.000 1.000 25 25 (0 salah)

4. Interpretasi Hasil & Implementasi Deployment
     1. Interpretasi Makna Metrik: Karena distribusi jumlah sampel antarspesies tidak seimbang (Chinstrap hanya berjumlah sekitar separuh dari Adelie), metrik Macro F1-Score dan Balanced Accuracy menjadi tolok ukur utama di samping Accuracy biasa. Nilai sempurna (1.000) pada seluruh metrik membuktikan bahwa model tidak bias terhadap kelas mayoritas dan mampu mengklasifikasikan ketiga spesies tanpa satu pun kesalahan prediksi (False Positive maupun False Negative bernilai nol).

     2. Dampak Rekayasa Fitur: Kombinasi fitur fisik dasar (flipper_length_mm, body_mass_g), lokasi pulau (island), serta fitur turunan bill_ratio membentuk batas keputusan (decision boundary) yang sangat jelas dan terpisah secara linear, menjelaskan mengapa model linear seperti Logistic Regression mampu mencapai performa optimal.

     3. Deployment Aplikasi: Pipeline lengkap (mencakup Custom Transformer PenguinFeatureEngineer, ColumnTransformer, dan model Logistic Regression) beserta metadata kelas disimpan ke dalam models/penguin_pipeline.joblib. Aplikasi antarmuka berbasis Streamlit (app.py) berhasil diintegrasikan sehingga pengguna dapat memasukkan pengukuran fisik mentah dan memperoleh prediksi spesies beserta distribusi probabilitas secara real-time.

In [1]:
!pip install -q pyngrok streamlit

Muat data

In [2]:
import os

DATA_PATH = "data/kasus03_penguins.csv"
os.makedirs("data", exist_ok=True)

if not os.path.exists(DATA_PATH):
    from google.colab import files
    print("Pilih file CSV kecelakaan dari komputer Anda...")
    unggahan = files.upload()
    nama_asli = next(iter(unggahan))
    os.replace(nama_asli, DATA_PATH)

print("Data siap:", DATA_PATH)

Pilih file CSV kecelakaan dari komputer Anda...


Saving kasus03_penguins.csv to kasus03_penguins.csv
Data siap: data/kasus03_penguins.csv


Eksplorasi awal

In [3]:
import pandas as pd

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("\nMissing value per kolom:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplikat species:", df.duplicated(subset="species").sum())
print("\nDistribusi target:")
print(df["species"].value_counts())
df.head()

Shape: (344, 8)

Missing value per kolom:
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

Duplikat species: 341

Distribusi target:
species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [4]:
%%writefile fitur.py
# fitur.py
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

def audit_data(df: pd.DataFrame) -> pd.DataFrame:
    """Melakukan audit dasar pada data mentah."""
    print(f"Jumlah baris awal: {len(df)}")
    print(f"Missing values per kolom:\n{df.isnull().sum()}")
    return df

class PenguinFeatureEngineer(BaseEstimator, TransformerMixin):
    """Transformer kustom untuk Feature Engineering sesuai aturan pipeline."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()

        # 1. Membuat fitur baru: Rasio paruh (Bill Ratio = Panjang Paruh / Kedalaman Paruh)
        if 'bill_length_mm' in X_copy.columns and 'bill_depth_mm' in X_copy.columns:
            X_copy['bill_ratio'] = X_copy['bill_length_mm'] / X_copy['bill_depth_mm']

        # 2. Membuang kolom yang tidak relevan: year
        if 'year' in X_copy.columns:
            X_copy = X_copy.drop(columns=['year'])

        return X_copy

Writing fitur.py


In [5]:
%%writefile latih.py
# latih.py
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from fitur import audit_data, PenguinFeatureEngineer

def main():
    # 1. Load Dataset
    data_path = 'data/kasus03_penguins.csv'
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"File {data_path} tidak ditemukan.")

    df = pd.read_csv(data_path)
    df = audit_data(df)

    X = df.drop(columns=['species'])
    y = df['species']

    # 2. Pembagian Data Train dan Test SEBELUM preprocessing/imputasi (Aturan Wajib)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # 3. Definisi Fitur Numerik & Kategorikal setelah Feature Engineering
    numeric_features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'bill_ratio']
    categorical_features = ['island', 'sex']

    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ])

    # 4. Definisi Dua Algoritma untuk Dibandingkan
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42)
    }

    best_model_name = None
    best_score = 0.0
    best_pipeline = None

    print("\n--- Perbandingan Model dengan Cross-Validation (Data Train) ---")
    for name, model in models.items():
        pipeline = Pipeline(steps=[
            ('feature_engineer', PenguinFeatureEngineer()),
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])

        # Cross-validation 5-fold
        cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
        mean_cv = cv_scores.mean()
        print(f"{name} - Mean CV Accuracy: {mean_cv:.4f} (+/- {cv_scores.std():.4f})")

        if mean_cv > best_score:
            best_score = mean_cv
            best_model_name = name
            best_pipeline = pipeline

    print(f"\nModel Terbaik Berdasarkan CV: {best_model_name} dengan Akurasi {best_score:.4f}")

    # 5. Training final pada seluruh data train dan evaluasi SATU KALI pada data test
    best_pipeline.fit(X_train, y_train)
    y_pred = best_pipeline.predict(X_test)

    test_accuracy = accuracy_score(y_test, y_pred)
    print(f"\nEvaluasi pada Data Test:")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # 6. Simpan Model ke folder models/ menggunakan joblib (Tanpa path absolut)
    os.makedirs('models', exist_ok=True)
    model_path = os.path.join('models', 'penguin_pipeline.joblib')

    # Simpan pipeline beserta metadata kelas target
    artifact = {
        'pipeline': best_pipeline,
        'classes': list(best_pipeline.classes_),
        'metrics': {
            'test_accuracy': test_accuracy,
            'best_model': best_model_name
        }
    }
    joblib.dump(artifact, model_path)
    print(f"\nPipeline berhasil disimpan di {model_path}")

if __name__ == '__main__':
    main()

Writing latih.py


In [6]:
%%writefile app.py
# app.py
import streamlit as st
import pandas as pd
import os
import joblib
import plotly.express as px

st.set_page_config(page_title="Klasifikasi Spesies Penguin", page_icon="🐧", layout="wide")

@st.cache_resource
def load_model():
    model_path = os.path.join('models', 'penguin_pipeline.joblib')
    if os.path.exists(model_path):
        return joblib.load(model_path)
    return None

artifact = load_model()

# Header Utama
st.title("Prediksi Spesies Penguin Antarktika")
st.markdown("Aplikasi *Machine Learning* interaktif untuk mengklasifikasikan spesies penguin (*Adelie*, *Chinstrap*, *Gentoo*) berdasarkan ciri fisiknya.")

if artifact is None:
    st.error("Model belum ditemukan! Harap jalankan skrip `!python latih.py` terlebih dahulu.")
else:
    pipeline = artifact['pipeline']
    metrics = artifact['metrics']

    # Sidebar untuk Input Parameter
    st.sidebar.header("Panel Kontrol Karakteristik")
    island = st.sidebar.selectbox("Pulau Pengamatan (Island)", ["Biscoe", "Dream", "Torgersen"])
    sex = st.sidebar.sidebar.selectbox("Jenis Kelamin (Sex)", ["male", "female"]) if "sex" in st.sidebar.__dir__() else st.sidebar.selectbox("Jenis Kelamin (Sex)", ["male", "female"])

    bill_length_mm = st.sidebar.slider("Panjang Paruh / Bill Length (mm)", 30.0, 60.0, 40.0)
    bill_depth_mm = st.sidebar.slider("Kedalaman Paruh / Bill Depth (mm)", 13.0, 22.0, 17.0)
    flipper_length_mm = st.sidebar.slider("Panjang Sirip / Flipper Length (mm)", 170.0, 240.0, 200.0)
    body_mass_g = st.sidebar.slider("Massa Tubuh / Body Mass (gram)", 2700.0, 6300.0, 4000.0)

    # Menyiapkan data input
    input_data = pd.DataFrame([{
        'island': island,
        'bill_length_mm': bill_length_mm,
        'bill_depth_mm': bill_depth_mm,
        'flipper_length_mm': flipper_length_mm,
        'body_mass_g': body_mass_g,
        'sex': sex,
        'year': 2008
    }])

    # Layout Utama Menjadi 2 Kolom
    col1, col2 = st.columns([1, 1])

    with col1:
        st.subheader("Ringkasan Parameter Input")
        st.dataframe(input_data.drop(columns=['year']), use_container_width=True)

        predict_btn = st.button("🔍 Jalankan Prediksi Spesies", type="primary", use_container_width=True)

    with col2:
        st.subheader("Hasil & Analisis Probabilitas")
        if predict_btn:
            prediction = pipeline.predict(input_data)[0]
            probabilities = pipeline.predict_proba(input_data)[0]

            st.success(f"### Spesies Terprediksi: **{prediction}** 🐧")

            # Membuat Bar Chart interaktif dengan Plotly yang memiliki animasi transisi
            prob_df = pd.DataFrame({
                'Spesies': pipeline.classes_,
                'Probabilitas': probabilities * 100
            })

            fig = px.bar(
                prob_df,
                x='Spesies',
                y='Probabilitas',
                text_auto='.1f',
                color='Spesies',
                color_discrete_sequence=px.colors.qualitative.Pastel,
                title="Tingkat Keyakinan Model (%)"
            )
            fig.update_layout(yaxis_title="Probabilitas (%)", showlegend=False, height=300)
            st.plotly_chart(fig, use_container_width=True)
        else:
            st.info("Silakan atur parameter di sidebar lalu klik tombol **Jalankan Prediksi Spesies**.")

    st.markdown("---")
    st.info(f"**Informasi Model**: Menggunakan algoritma **{metrics['best_model']}** dengan akurasi pengujian data test sebesar **{metrics['test_accuracy']*100:.2f}%**.")

Writing app.py


In [7]:
!python latih.py

Jumlah baris awal: 344
Missing values per kolom:
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
year                  0
dtype: int64

--- Perbandingan Model dengan Cross-Validation (Data Train) ---
Logistic Regression - Mean CV Accuracy: 0.9927 (+/- 0.0089)
Random Forest - Mean CV Accuracy: 0.9891 (+/- 0.0089)

Model Terbaik Berdasarkan CV: Logistic Regression dengan Akurasi 0.9927

Evaluasi pada Data Test:
Test Accuracy: 1.0000

Classification Report:
              precision    recall  f1-score   support

      Adelie       1.00      1.00      1.00        30
   Chinstrap       1.00      1.00      1.00        14
      Gentoo       1.00      1.00      1.00        25

    accuracy                           1.00        69
   macro avg       1.00      1.00      1.00        69
weighted avg       1.00      1.00      1.00        69


Pipeline berhasil disimpan di models/pen

In [8]:
!streamlit run app.py --server.headless true --server.port 8501 > /content/streamlit.log 2>&1 &

In [9]:
from pyngrok import ngrok
import os

# Masukkan token autentikasi ngrok Anda di bawah ini
# Ambil dari: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "3MH31SmFzzeqObNuIKVO13JQdpr_6V739oYCcJdKWj3z8v7PK"

try:
    ngrok.set_auth_token(NGROK_TOKEN)
except Exception as e:
    # Jika token sudah ada atau terjadi error penyimpanan, kita tampilkan log dan lanjutkan
    print(f"Catatan konfigurasi token: {e}")

# Tutup koneksi ngrok sebelumnya jika ada
ngrok.kill()

# Buat link publik baru
public_url = ngrok.connect(8501)
print("🔗 Klik link aplikasi Streamlit Anda di bawah ini:")
print(public_url)

🔗 Klik link aplikasi Streamlit Anda di bawah ini:
NgrokTunnel: "https://imperial-safely-prototype.ngrok-free.dev" -> "http://localhost:8501"
